# Run QPT for DakisX on real IQM hardware

Adapted from `QPT_X.ipynb`, compatible with the `*_calib.json` + `*_samples.npz` pair
saved by `X_calibration.ipynb` (`Results/Calibration/scqc_x_*_calib.json`).

**This notebook submits real jobs to IQM hardware and consumes real shot budget.**
Requires `IQM_TOKEN` as an environment variable (see `SX_calibration_exp.ipynb`) and the
same hardware-SDK packages as the other notebooks here.

## How this differs from `QPT_SX_exp.ipynb`

- **Gate class**: `DakisXGate` subclasses the general `PRX_CustomWaveforms` (no
  `rz_before`/`rz_after` -- see `X_calibration.ipynb`'s intro for why), not the
  SX-specific class. Field names from the calibration file are `amp_x` (not `amp_sx`)
  and there's no `rz_after` to read.
- **A real intrinsic virtual-Z artifact, discovered empirically on hardware**: when a
  standalone DakisX gate is calibrated alone (as `X_calibration.ipynb` does -- a single
  isolated gate, nothing before or after it), there's nothing for a Z-basis population
  measurement to see if its phase axis is off (verified in `X_calibration.ipynb`'s
  design phase: a lone `pi` rotation inverts population regardless of axis). But once
  DakisX is *composed* with other, natively-compiled gates in the same circuit -- which
  every QPT circuit does (prep gate, DakisX, measurement-basis gate) -- the two gate
  representations don't share the same phase frame, and DakisX behaves like
  `Rz(theta) . X` for some small intrinsic `theta`. This shows up as reduced process
  fidelity that a standalone calibration can't catch.
  This notebook extracts `theta` directly from the QPT data itself (no extra hardware
  runs needed): the equatorial input states `|+>` and `|+i>` *do* have coherence, so
  their measured populations after DakisX reveal the axis error the same way the
  computational-basis states can't. It then re-runs QPT with every post-DakisX gate's
  phase shifted by `+theta` to compensate, and reports both the uncorrected and
  corrected fidelity so you can see the effect.
- **Circuit compilation for the `dakis` arm is a two-step process**
  (`transpile_to_IQM` -> tag the specific PRX instruction between the QPT barriers as
  `dakis_x` -> optionally shift every *subsequent* PRX phase by the VZ correction),
  instead of the simpler `s.gate_definitions.prx.default_implementation = ...` override
  used for standalone circuits -- needed precisely because this notebook's circuits
  contain other native gates whose phase interacts with DakisX's.
- **`Delta_ghz` is read from the calibration and used**, not forced to zero. The
  original reference notebook had `Delta_ghz = calib.get('Delta_ghz', 0.0) * 0` (an
  explicit override, presumably situational for that specific run) -- since
  `X_calibration.ipynb` does calibrate a real `Delta_ghz`, this notebook uses it.
  Set it to `0.0` yourself below if you want the reference's original behavior.
- **`run_qpt_batched` is carried over but the reference notebook itself flags it as
  untested on real hardware** for the `dakis` path -- this notebook uses the safer
  per-circuit `run_qpt` (which can abort before spending QPT shots if REM looks bad)
  for the `dakis` arm, and only uses the batched version for `native_fixed`, matching
  how the reference notebook actually used them.


In [ ]:
import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from datetime import datetime

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Choi, process_fidelity
from qiskit.circuit.library import XGate

from iqm.qiskit_iqm import IQMProvider, transpile_to_IQM
from iqm.pulla.pulla import Pulla
from iqm.pulla.utils_qiskit import get_qiskit_compiler, qiskit_circuits_to_pulla
from iqm.pulse.gates.prx import PRX_CustomWaveforms
from iqm.pulse.playlist.waveforms import Waveform

import iqm_tools


## Connect to IQM hardware

Everything from here on talks to a live server.


In [ ]:
# IQM_TOKEN must be set as an environment variable before running this notebook.
# Never hardcode a real token in a notebook cell, especially in a public repo.
#   export IQM_TOKEN="<your-api-token-here>"   # set this in your shell, before launching Jupyter
if 'IQM_TOKEN' not in os.environ:
    raise RuntimeError(
        "Set the IQM_TOKEN environment variable before running this notebook "
        "(e.g. `export IQM_TOKEN=<your-api-token-here>` in your shell before launching Jupyter). "
        "Never hardcode a token in a notebook cell, especially in a public repo."
    )

iqm_server_url = 'https://resonance.iqm.tech/emerald'   # <- change to your target machine

pulla   = Pulla(iqm_server_url)
backend = IQMProvider(iqm_server_url).get_backend()

print(f'Connected to {iqm_server_url}')


## Load the calibration

Compatible with the `*_calib.json` + `*_samples.npz` pair saved by `X_calibration.ipynb`.


In [ ]:
CAL_DIR = 'Results/Calibration'

calib_files = sorted(glob.glob(os.path.join(CAL_DIR, 'scqc_x_*_calib.json')), key=os.path.getmtime)
print(f'{len(calib_files)} calibration file(s) found:\n')
for i, f in enumerate(calib_files):
    print(f'[{i}]  {os.path.basename(f)}')

idx = -1   # -1 = most recent
calib_path = calib_files[idx]

with open(calib_path) as f:
    calib = json.load(f)

samples_path = os.path.join(os.path.dirname(os.path.abspath(calib_path)), calib['samples_file'])
samples   = np.load(samples_path)
i_samples = samples['i_samples']
q_samples = samples['q_samples']

qbt       = calib['qubit']
Texp      = calib['Texp_ns']
Delta_ghz = calib.get('Delta_ghz', 0.0)   # reference notebook forced this to 0.0 -- see intro

print(f"\nCalibration: {qbt}, Texp={Texp} ns, Delta={Delta_ghz*1e3:+.3f} MHz")
print(f"  amp_x={calib['amp_x']:.5f}")

_I_SAMPLES = i_samples.copy()
_Q_SAMPLES = q_samples.copy()
_N         = len(_I_SAMPLES)

@dataclass(frozen=True)
class DakisXWave_I(Waveform):
    def _sample(self, sample_coords: np.ndarray) -> np.ndarray:
        t_grid = np.linspace(0, 1, _N, endpoint=False)
        return np.interp(sample_coords + 0.5, t_grid, _I_SAMPLES,
                         left=_I_SAMPLES[0], right=_I_SAMPLES[-1])

@dataclass(frozen=True)
class DakisXWave_Q(Waveform):
    def _sample(self, sample_coords: np.ndarray) -> np.ndarray:
        t_grid = np.linspace(0, 1, _N, endpoint=False)
        return np.interp(sample_coords + 0.5, t_grid, _Q_SAMPLES,
                         left=_Q_SAMPLES[0], right=_Q_SAMPLES[-1])

class DakisXGate(PRX_CustomWaveforms, wave_i=DakisXWave_I, wave_q=DakisXWave_Q):
    """Leakage-suppressed X (pi) gate."""
    pass

compiler = get_qiskit_compiler(pulla, backend)
compiler.add_implementation('prx', 'dakis_x', DakisXGate)

print('DakisXGate ready.')


## Fix the native gate's current calibration

Reads the native implementation's *current* Settings and freezes them, so both arms of
the comparison stay fixed for the whole QPT run even if the live calibration set changes.


In [ ]:
_qc_ref      = QuantumCircuit(1); _qc_ref.x(0)
_s_ref       = compiler.get_settings(circuits=[_qc_ref])
_native_impl = _s_ref.gate_definitions.prx.default_implementation.value
_prx_node    = _s_ref.gates.prx[_native_impl][qbt]

fixed_native_params = {
    'implementation': _native_impl,
    'duration':       _prx_node.duration.value,
    'amplitude_i':    _prx_node.amplitude_i.value,
    'amplitude_q':    _prx_node.amplitude_q.value,
    'rz_before':      _prx_node.rz_before.value,
    'rz_after':       _prx_node.rz_after.value,
    'full_width':     _prx_node.full_width.value,
    'center_offset':  _prx_node.center_offset.value,
}

def get_fixed_native_settings(circuits):
    s    = compiler.get_settings(circuits=circuits)
    impl = fixed_native_params['implementation']
    node = s.gates.prx[impl][qbt]
    node.duration      = fixed_native_params['duration']
    node.amplitude_i   = fixed_native_params['amplitude_i']
    node.amplitude_q   = fixed_native_params['amplitude_q']
    node.rz_before     = fixed_native_params['rz_before']
    node.rz_after      = fixed_native_params['rz_after']
    node.full_width    = fixed_native_params['full_width']
    node.center_offset = fixed_native_params['center_offset']
    return s

# duration shown is the underlying SX (pi/2) pulse; native X = 2 x this
print(f'Native impl: {_native_impl}  (SX duration={fixed_native_params["duration"]*1e9:.0f} ns  ->  X = 2xSX)')


## QPT circuit structure

```
prep_gate | barrier | x (TARGET) | barrier | meas_basis_gate | measure
```

- 4 input states: |0> (p0), |1> (p1), |+>=H|0> (p2), |+i>=S.H|0> (p3)
- 3 measurement bases: Z (m0), X=H.Z (m1), Y=H.Sdg.Z (m2) -> P(|1>) gives the Bloch vector of each output state
- + 2 REM circuits (|0> and |1> prepared, then measured) for readout calibration. The
  |1> REM circuit uses the same barrier-wrapped X structure as the QPT circuits, so
  'dakis' mode applies DakisX there too -- keeping REM and QPT consistent.


In [ ]:
def make_qpt_circuits():
    records = []

    for p_idx in range(4):
        for m_idx in range(3):
            qc = QuantumCircuit(1, 1)

            # State preparation
            if   p_idx == 1: qc.x(0)             # |1>
            elif p_idx == 2: qc.h(0)             # |+>
            elif p_idx == 3: qc.h(0); qc.s(0)    # |+i>
            # p_idx == 0: nothing -> |0>

            qc.barrier(0)
            qc.x(0)           # <- TARGET GATE
            qc.barrier(0)

            # Measurement basis rotation
            if   m_idx == 1: qc.h(0)             # X basis
            elif m_idx == 2: qc.sdg(0); qc.h(0)  # Y basis
            # m_idx == 0: nothing -> Z basis

            qc.measure(0, 0)
            qc.name = f'qpt_p{p_idx}_m{m_idx}'
            records.append({
                'name':     qc.name,
                'metadata': {'kind': 'qpt', 'prep_index': p_idx, 'meas_index': m_idx},
                'circuit':  qc,
            })

    # Readout error mitigation circuits
    for true_state in (0, 1):
        qc = QuantumCircuit(1, 1)
        if true_state == 1:
            qc.barrier(0)
            qc.x(0)    # same TARGET structure so 'dakis' mode applies DakisX here too
            qc.barrier(0)
        qc.measure(0, 0)
        qc.name = f'rem_{true_state}'
        records.append({
            'name':     qc.name,
            'metadata': {'kind': 'rem', 'true_state': true_state},
            'circuit':  qc,
        })

    return records

records = make_qpt_circuits()

print(f'{len(records)} circuits total ({len(records)-2} QPT + 2 REM)\n')
for r in records:
    print(r['name'], ':', r['metadata'])


## Run QPT

In [ ]:
def run_qpt(records, gate, n_shots, rem_p01_max=0.15, vz_correction=0.0):
    """
    Run all QPT+REM circuits on IQM and return results.
    gate: 'dakis' | 'native_fixed' | 'native'
    vz_correction: phase shift (rad) added to every post-DakisX PRX gate.
                   Pass the theta extracted by extract_vz_from_qpt() to undo
                   DakisX's intrinsic Rz(theta) rotation.
    """
    assert gate in ('dakis', 'native_fixed', 'native')
    corr_str = f'  vz_correction={np.degrees(vz_correction):+.2f} deg' if vz_correction else ''
    print(f'Running QPT  [{gate}]  {n_shots} shots/circuit{corr_str} ...')

    def _compile_dakis(qc):
        """Two-step compile: Qiskit->IQM (pinned to qbt), tag target PRX, compile."""
        transpiled = transpile_to_IQM(
            qc, backend,
            restrict_to_qubits=[qbt],
            optimize_single_qubits=True,
            ignore_barriers=False,
        )
        iqm_circs = qiskit_circuits_to_pulla([transpiled], qubit_idx_to_name={0: qbt})
        c2 = pulla.get_standard_compiler(exa_style_pp=False)
        c2.add_implementation('prx', 'dakis_x', DakisXGate)
        # Tag the PRX between the two QPT barriers as dakis_x, then shift every
        # subsequent PRX phase by +vz_correction to compensate for DakisX's
        # intrinsic Rz(theta) rotation.
        barrier_count = 0
        dakis_found   = False
        for inst in iqm_circs[0].instructions:
            if inst.name == 'barrier':
                barrier_count += 1
            elif barrier_count == 1 and inst.name == 'prx' and not dakis_found:
                inst.implementation = 'dakis_x'
                dakis_found = True
            elif dakis_found and inst.name == 'prx' and vz_correction != 0.0:
                inst.args['phase'] = inst.args.get('phase', 0.0) + vz_correction
        s    = c2.get_settings(circuits=iqm_circs)
        node = s.gates.prx.dakis_x[qbt]
        node.duration    = Texp * 1e-9
        node.amplitude_i = calib['amplitude_i']
        node.amplitude_q = calib['amplitude_q']
        s.set_shots(n_shots)
        return c2.compile(iqm_circs, settings=s)

    def run_one(r):
        qc = r['circuit']
        if gate == 'dakis':
            jd, ctx = _compile_dakis(qc)
        else:
            if gate == 'native_fixed':
                s = get_fixed_native_settings([qc])
            else:
                s = compiler.get_settings(circuits=[qc])
            s.set_shots(n_shots)
            jd, ctx = compiler.compile(circuits=[qc], components=[qbt], settings=s)
        job = pulla.submit_playlist(jd, context=ctx)
        job.wait_for_completion()
        if job.status != 'completed':
            print(f'  {r["name"]}: FAILED ({job.status})')
            return {**r, 'prob_raw': float('nan'), 'prob_corr': float('nan')}
        res       = job.result(compiler)
        prob_corr = float(res.dataset[f'{qbt}__c_1_0_0_excited_state_probability'].values.flat[0])
        prob_raw  = float(res.dataset['counter.result'].values.flat[-1])
        print(f'  {r["name"]:20s}  raw={prob_raw:.4f}  corr={prob_corr:.4f}')
        return {**r, 'prob_raw': prob_raw, 'prob_corr': prob_corr}

    rem_records = [r for r in records if r['metadata'].get('kind') == 'rem']
    qpt_records = [r for r in records if r['metadata'].get('kind') == 'qpt']

    # REM first -- abort before spending QPT shots if readout looks bad
    print('  [REM]')
    results = [run_one(r) for r in rem_records]

    rem0 = next(r for r in results if r['metadata'].get('true_state') == 0)
    rem1 = next(r for r in results if r['metadata'].get('true_state') == 1)
    p10  = rem0['prob_raw']
    p01  = 1 - rem1['prob_raw']
    print(f'  REM check:  p10={p10:.4f}  p01={p01:.4f}', end='')

    if p01 > rem_p01_max:
        print(f'  <- BAD (>{rem_p01_max:.2f}) -- aborting before QPT jobs')
        raise ValueError(
            f'p01={p01:.4f} exceeds threshold {rem_p01_max}. '
            'Check gate calibration (wrong amplitude?).'
        )
    print('  OK')

    print('  [QPT]')
    results += [run_one(r) for r in qpt_records]

    return results


In [ ]:
def run_qpt_batched(records, gate, n_shots, rem_p01_max=0.15, vz_correction=0.0):
    """
    Same as run_qpt, but compiles + submits ALL circuits (REM + QPT) in ONE
    playlist instead of one at a time -- one job submission instead of 14.
    gate: 'dakis' | 'native_fixed' | 'native'

    TRADEOFF vs run_qpt: REM can no longer abort BEFORE spending shots on the
    QPT circuits (everything is submitted together) -- it still raises
    afterward if REM looks bad, but you don't save those shots in that case.

    UNTESTED on real hardware for gate='dakis' (per the reference notebook this
    was ported from) -- this notebook only uses it for gate='native_fixed';
    inspect the printed dataset dims/shape before trusting the dakis path.
    """
    assert gate in ('dakis', 'native_fixed', 'native')
    corr_str = f'  vz_correction={np.degrees(vz_correction):+.2f} deg' if vz_correction else ''
    print(f'Running batched QPT  [{gate}]  {n_shots} shots/circuit{corr_str} ...')

    qcs = [r['circuit'] for r in records]

    if gate == 'dakis':
        iqm_circs_all = []
        for qc in qcs:
            transpiled = transpile_to_IQM(
                qc, backend, restrict_to_qubits=[qbt],
                optimize_single_qubits=True, ignore_barriers=False,
            )
            iqm_circ = qiskit_circuits_to_pulla([transpiled], qubit_idx_to_name={0: qbt})[0]
            barrier_count = 0
            dakis_found = False
            for inst in iqm_circ.instructions:
                if inst.name == 'barrier':
                    barrier_count += 1
                elif barrier_count == 1 and inst.name == 'prx' and not dakis_found:
                    inst.implementation = 'dakis_x'
                    dakis_found = True
                elif dakis_found and inst.name == 'prx' and vz_correction != 0.0:
                    inst.args['phase'] = inst.args.get('phase', 0.0) + vz_correction
            iqm_circs_all.append(iqm_circ)

        c2 = pulla.get_standard_compiler(exa_style_pp=False)
        c2.add_implementation('prx', 'dakis_x', DakisXGate)
        s = c2.get_settings(circuits=iqm_circs_all)
        node = s.gates.prx.dakis_x[qbt]
        node.duration    = Texp * 1e-9
        node.amplitude_i = calib['amplitude_i']
        node.amplitude_q = calib['amplitude_q']
        s.set_shots(n_shots)
        jd, ctx = c2.compile(iqm_circs_all, settings=s)
    else:
        if gate == 'native_fixed':
            s = get_fixed_native_settings(qcs)
        else:
            s = compiler.get_settings(circuits=qcs)
        s.set_shots(n_shots)
        jd, ctx = compiler.compile(circuits=qcs, components=[qbt], settings=s)

    job = pulla.submit_playlist(jd, context=ctx)
    job.wait_for_completion()
    if job.status != 'completed':
        raise RuntimeError(f'Batched QPT job failed: {job.status}')
    res = job.result(compiler)

    print('  dataset dims/shape:', res.dataset['counter.result'].dims, res.dataset['counter.result'].shape)

    results = []
    for i, r in enumerate(records):
        prob_raw  = float(res.dataset['counter.result'].isel(circuit_index=i).values.flat[-1])
        prob_corr = float(res.dataset[f'{qbt}__c_1_0_0_excited_state_probability'].isel(circuit_index=i).values.flat[0])
        print(f'  {r["name"]:20s}  raw={prob_raw:.4f}  corr={prob_corr:.4f}')
        results.append({**r, 'prob_raw': prob_raw, 'prob_corr': prob_corr})

    rem0 = next(rr for rr in results if rr['metadata'].get('true_state') == 0)
    rem1 = next(rr for rr in results if rr['metadata'].get('true_state') == 1)
    p10 = rem0['prob_raw']; p01 = 1 - rem1['prob_raw']
    print(f'  REM check:  p10={p10:.4f}  p01={p01:.4f}', end='')
    if p01 > rem_p01_max:
        print(f'  <- BAD (>{rem_p01_max:.2f})')
        raise ValueError(f'p01={p01:.4f} exceeds threshold {rem_p01_max} '
                          f'(after spending all shots -- batching cannot abort early).')
    print('  OK')

    return results


In [ ]:
n_shots = 3000

results_dakis = run_qpt(records, gate='dakis', n_shots=n_shots)


## Characterise the DakisX virtual-Z angle

Effective gate: `DakisX ~= Rz(theta) . X`

For input `|+>` = (1, 0, 0) Bloch: `output = Rz(theta)|+> = (cos theta, sin theta, 0)`
- X-meas `P(1) = (1 - cos theta)/2` -> `cos theta = 1 - 2*P(qpt_p2_m1)`
- Y-meas `P(1) = (1 - sin theta)/2` -> `sin theta = 1 - 2*P(qpt_p2_m2)`

For input `|+i>` = (0, 1, 0) Bloch (independent cross-check): `X|+i> = |-i> = (0,-1,0)`,
then `Rz(theta) -> (sin theta, -cos theta, 0)`
- X-meas `P(1) = (1 - sin theta)/2` -> `sin theta = 1 - 2*P(qpt_p3_m1)`
- Y-meas `P(1) = (1 + cos theta)/2` -> `cos theta = 2*P(qpt_p3_m2) - 1`

No new hardware runs needed -- the QPT data collected above already contains all the
information. `run_vz_characterisation()` below is an optional dedicated, higher-shot
measurement of the same quantity, for a cleaner estimate.


In [ ]:
def extract_vz_from_qpt(results):
    """Compute DakisX VZ angle theta from two equatorial QPT inputs."""
    r = {x['name']: x for x in results}

    cos_t2   = 1 - 2 * r['qpt_p2_m1']['prob_raw']
    sin_t2   = 1 - 2 * r['qpt_p2_m2']['prob_raw']
    theta_p2 = np.arctan2(sin_t2, cos_t2)

    sin_t3   =  1 - 2 * r['qpt_p3_m1']['prob_raw']
    cos_t3   = -(1 - 2 * r['qpt_p3_m2']['prob_raw'])
    theta_p3 = np.arctan2(sin_t3, cos_t3)

    theta_avg = (theta_p2 + theta_p3) / 2
    print('DakisX VZ angle from QPT data:')
    print(f'  |+>  input:  theta = {np.degrees(theta_p2):+.2f} deg  ({theta_p2:.5f} rad)')
    print(f'  |+i> input:  theta = {np.degrees(theta_p3):+.2f} deg  ({theta_p3:.5f} rad)')
    print(f'  average:     theta = {np.degrees(theta_avg):+.2f} deg  ({theta_avg:.5f} rad)')
    return theta_avg


def run_vz_characterisation(n_shots=4000):
    """Dedicated 2-circuit measurement of the DakisX VZ angle (optional, higher-precision)."""
    print(f'VZ characterisation  [{n_shots} shots/circuit] ...')
    P = {}
    for label, add_sdg in (('X', False), ('Y', True)):
        qc = QuantumCircuit(1, 1)
        qc.h(0); qc.barrier(0); qc.x(0); qc.barrier(0)
        if add_sdg:
            qc.sdg(0)
        qc.h(0); qc.measure(0, 0); qc.name = f'vz_{label}'

        transpiled = transpile_to_IQM(qc, backend, restrict_to_qubits=[qbt],
                                      optimize_single_qubits=True, ignore_barriers=False)
        iqm_circs  = qiskit_circuits_to_pulla([transpiled], qubit_idx_to_name={0: qbt})
        c2 = pulla.get_standard_compiler(exa_style_pp=False)
        c2.add_implementation('prx', 'dakis_x', DakisXGate)
        barrier_count = 0
        for inst in iqm_circs[0].instructions:
            if inst.name == 'barrier':
                barrier_count += 1
            elif barrier_count == 1 and inst.name == 'prx':
                inst.implementation = 'dakis_x'; break
        s = c2.get_settings(circuits=iqm_circs)
        node = s.gates.prx.dakis_x[qbt]
        node.duration    = Texp * 1e-9
        node.amplitude_i = calib['amplitude_i']
        node.amplitude_q = calib['amplitude_q']
        s.set_shots(n_shots)
        jd, ctx = c2.compile(iqm_circs, settings=s)
        job = pulla.submit_playlist(jd, context=ctx)
        job.wait_for_completion()
        prob = float(job.result(compiler).dataset['counter.result'].values.flat[-1])
        print(f'  {qc.name}: P(1) = {prob:.4f}')
        P[label] = prob

    cos_theta = 1 - 2 * P['X']
    sin_theta = 1 - 2 * P['Y']
    theta     = np.arctan2(sin_theta, cos_theta)
    print(f'  -> theta = {np.degrees(theta):+.2f} deg = {theta:.5f} rad')
    return theta


# compute theta from the QPT data already collected above -- no new hardware jobs
dakis_vz_rad = extract_vz_from_qpt(results_dakis)
print(f'\ndakis_vz_rad = {dakis_vz_rad:.5f} rad')


## Re-run QPT with the VZ correction applied

Optional: uncomment below to also collect a dedicated, higher-shot VZ measurement and
use that instead of `dakis_vz_rad`.


In [ ]:
# theta_vz = run_vz_characterisation(n_shots=4000)
theta_vz = dakis_vz_rad

results_dakis_corr = run_qpt(records, gate='dakis', n_shots=n_shots,
                              vz_correction=theta_vz)


In [ ]:
results_native = run_qpt_batched(records, gate='native_fixed', n_shots=n_shots)


## Process reconstruction

Each circuit measures P(|1>) for one (prep, basis) pair. From 3 measurements per input
state we get the full output Bloch vector, from which the Choi matrix is assembled.
Identical math to `QPT_SX_exp.ipynb`, verified there against real saved data.


In [ ]:
I2 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)    # Pauli X
sy = np.array([[0, -1j], [1j, 0]], dtype=complex) # Pauli Y
sz = np.array([[1, 0], [0, -1]], dtype=complex)   # Pauli Z

def rem_correct(prob_raw, p10, p01):
    denom = 1 - p01 - p10
    if abs(denom) < 1e-6:
        return prob_raw
    return np.clip((prob_raw - p10) / denom, 0, 1)

def reconstruct_choi(results, use_corrected=True):
    prob_key = 'prob_corr' if use_corrected else 'prob_raw'

    rem0 = next(r for r in results if r['metadata'].get('true_state') == 0)
    rem1 = next(r for r in results if r['metadata'].get('true_state') == 1)
    p10  = rem0[prob_key]
    p01  = 1 - rem1[prob_key]
    print(f'  REM: p10={p10:.4f}  p01={p01:.4f}')

    p = np.full((4, 3), np.nan)
    for r in results:
        meta = r['metadata']
        if meta.get('kind') != 'qpt':
            continue
        j, k = meta['prep_index'], meta['meas_index']
        p[j, k] = rem_correct(r[prob_key], p10, p01)

    sigma = []
    for j in range(4):
        x_exp = 1 - 2 * p[j, 1]
        y_exp = 1 - 2 * p[j, 2]
        z_exp = 1 - 2 * p[j, 0]
        sigma.append((I2 + x_exp*sx + y_exp*sy + z_exp*sz) / 2)

    A   = 2*sigma[2] - sigma[0] - sigma[1]
    B   = -1j * (2*sigma[3] - sigma[0] - sigma[1])
    L01 = (A - B) / 2
    L10 = (A + B) / 2

    C = np.block([[sigma[0], L01],
                  [L10,      sigma[1]]])

    return C, p, sigma

print('reconstruct_choi() ready')


In [ ]:
use_corrected = False   # set True to use readout-corrected probabilities throughout

ideal_choi = Choi(Operator(XGate()))
ideal_C    = np.array(ideal_choi.data)

print('-- DakisX (uncorrected) --')
C_uncorr, p_uncorr, _ = reconstruct_choi(results_dakis, use_corrected)
F_uncorr = process_fidelity(Choi(C_uncorr), ideal_choi).real
print(f'  F = {F_uncorr:.4f}')

print('-- DakisX (VZ-corrected) --')
C_corr, p_corr, _ = reconstruct_choi(results_dakis_corr, use_corrected)
F_corr = process_fidelity(Choi(C_corr), ideal_choi).real
print(f'  F = {F_corr:.4f}')

print('-- Native X --')
C_native, p_native, _ = reconstruct_choi(results_native, use_corrected)
F_native = process_fidelity(Choi(C_native), ideal_choi).real
print(f'  F = {F_native:.4f}')


In [ ]:
prep_labels = ['|0>', '|1>', '|+>', '|+i>']
meas_labels = ['Z', 'X', 'Y']

datasets = [('DakisX uncorr', p_uncorr), ('DakisX VZ-corr', p_corr), ('Native X', p_native)]

fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 4))

for ax, (label, p_data) in zip(axes, datasets):
    im = ax.imshow(p_data, vmin=0, vmax=1, cmap='RdBu_r', aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(meas_labels)
    ax.set_yticks(range(4)); ax.set_yticklabels(prep_labels)
    ax.set_xlabel('Measurement basis'); ax.set_ylabel('Input state')
    ax.set_title(label)
    plt.colorbar(im, ax=ax)
    for j in range(4):
        for k in range(3):
            ax.text(k, j, f'{p_data[j,k]:.3f}', ha='center', va='center', fontsize=8)

plt.suptitle(f'QPT P(|1>)  |  {qbt}  |  F_uncorr={F_uncorr:.4f}  F_corr={F_corr:.4f}  F_native={F_native:.4f}',
             fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
choi_datasets = [('Ideal X', ideal_C), ('DakisX (VZ-corr)', C_corr), ('Native (fixed)', C_native)]

n_cols = len(choi_datasets)
fig, axes = plt.subplots(2, n_cols, figsize=(5*n_cols, 8))

for col, (title, C) in enumerate(choi_datasets):
    for row, (part, label) in enumerate([(np.real, 'Re'), (np.imag, 'Im')]):
        ax = axes[row, col]
        im = ax.imshow(part(C), vmin=-0.5, vmax=0.5, cmap='RdBu')
        ax.set_title(f'{title}  ({label})')
        ax.set_xticks(range(4)); ax.set_yticks(range(4))
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f'Choi matrices  |  {qbt}  |  F_dakis={F_corr:.4f}  F_native={F_native:.4f}', fontsize=10)
plt.tight_layout()
plt.show()


## Save the result

In [ ]:
expr_params  = iqm_tools.get_qubit_params(iqm_server_url)
qbt_row      = expr_params[expr_params['qubit'] == qbt].iloc[0]
calib_set_id = expr_params.attrs.get('calibration_set_id', 'unknown')
machine      = iqm_server_url.rstrip('/').split('/')[-1]

def _serialise_results(results):
    return [
        {'name': r['name'], 'metadata': r['metadata'],
         'prob_raw': r['prob_raw'], 'prob_corr': r['prob_corr']}
        for r in results
    ]

record = {
    'timestamp':          datetime.now().isoformat(),
    'machine':            machine,
    'calibration_set_id': calib_set_id,
    'qubit':              qbt,
    'T1_us':              float(qbt_row['T1_us']),
    'T2_ramsey_us':       float(qbt_row['T2_ramsey_us']),
    'n_shots':            n_shots,
    'dakis_calibration':  calib,
    'dakis_vz_rad':       float(theta_vz),
    'native_gate':        fixed_native_params,
    'process_fidelity': {
        'dakis':          float(F_corr),
        'dakis_uncorr':   float(F_uncorr),
        'native':         float(F_native),
    },
    'choi_matrix': {
        'dakis_real':  C_corr.real.tolist(),
        'dakis_imag':  C_corr.imag.tolist(),
        'native_real': C_native.real.tolist(),
        'native_imag': C_native.imag.tolist(),
        'ideal_real':  ideal_C.real.tolist(),
        'ideal_imag':  ideal_C.imag.tolist(),
    },
    'results_dakis':          _serialise_results(results_dakis_corr),
    'results_dakis_uncorr':   _serialise_results(results_dakis),
    'results_native':         _serialise_results(results_native),
}

ts        = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir  = 'Results/QPT'
save_path = f'{save_dir}/qpt_x_{qbt}_{machine}_{ts}.json'
os.makedirs(save_dir, exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(record, f, indent=2)

print(f'Saved: {save_path}')
print(f'  F_dakis={F_corr:.4f}  (uncorrected: {F_uncorr:.4f})   F_native={F_native:.4f}')
